# Prompt Chaining Hands-On Code Example

The following code implements a two-step prompt chain that functions as a data processing pipeline. The initial stage is designed to parse unstructured text and extract specific information. The subsequent stage then receives this extracted output and transforms it into a structured data format.

In [ ]:
# !pip install langchain langchain-community langchain-google-genai langgraph

> Note: Create a `.env` file in the same directory with your Google Generative AI API key:
> ```
> GOOGLE_API_KEY="<your_google_api_key_here>"
> ```

In [ ]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
load_dotenv(override=True)

True

In [3]:
# Initialize the Language Model (using ChatOpenAI is recommended)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [4]:
# --- Prompt 1: Extract Information ---
prompt_extract = ChatPromptTemplate.from_template("Extract the technical specifications from the following text:\n\n{text_input}")

In [5]:
# --- Prompt 2: Transform to JSON ---
prompt_transform = ChatPromptTemplate.from_template("Transform the following spcecifications into a JSON object with 'cpu', 'memory', and 'storage' as keys:\n\n{specifications}")

In [6]:
# Build chain using LCEL
# The StrOutputParser converts the LLM's message output to a simple string.
extraction_chain = prompt_extract | llm | StrOutputParser()

In [7]:
# The full chain passes the output of the extraction chain into the 'specifications' variable for the transformation prompt.
full_chain = (
    {"specifications": extraction_chain}
    | prompt_transform
    | llm
    | StrOutputParser()
)

In [8]:
# Run the chain
input_text = "The new laptop model features a 3.5 GHz oct-core processor, 16 GB of RAM, and a 1TB NVMe SSD."

# Execute the chain with the input text dictionary
final_result = full_chain.invoke({"text_input": input_text})

In [9]:
print("\n--- Final JSON Output ---")
print(final_result)


--- Final JSON Output ---
```json
{
  "cpu": "3.5 GHz oct-core",
  "memory": "16 GB",
  "storage": "1TB NVMe SSD"
}
```


This Python code demonstrates how to use the LangChain library to process text. It utilizes two separate prompts: one to extract technical specifications from an input string and another to format these specifications into a JSON object. The ChatGoogleGenerativeAI model is employed for language model interactions, and the StrOutputParser ensures the output is in a usable string format. The LangChain Expression Language (LCEL) is used to elegantly chain these prompts and the language model together. The first chain, extraction_chain, extracts the specifications. The full_chain then takes the output of the extraction and uses it as input for the transformation prompt. A sample input text describing a laptop is provided. The full_chain is invoked with this text, processing it through both steps. The final result, a JSON string containing the extracted and formatted specifications, is then printed.